## Imports

In [1]:
#anaconda prompt > conda install -c anaconda openpyxl

import openpyxl

from bs4 import BeautifulSoup
import requests
import time
from datetime import datetime
import os



import warnings
warnings.filterwarnings('ignore')

## Funcion para extraccion de datos de Nike

In [2]:
# Conectar y obtener datos desde Nike

#Se obtiene el User Agent desde http://httpbin.org/get
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/108.0.0.0 Safari/537.36", "Accept-Encoding":"gzip, deflate", "Accept":"text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8", "DNT":"1","Connection":"close", "Upgrade-Insecure-Requests":"1"}

#Funcion para obtener el precio de un producto
def get_Nike_price(url):

    #Se obtiene el HTML del producto
    page = requests.get(url, headers=headers, verify=False)
    soup1 = BeautifulSoup(page.content, "html.parser")

    #Se extrae el precio del HTML
    try:
        price = soup1.find('div', {"class": "product-price es__styling is--current-price css-11s12ax"}).text.strip()
               
    except:
        price = soup1.find("div", {"class" : "product-price is--current-price css-1ydfahe"}).text.strip() 
                
                
    return price

def get_Nike_price2(url):

    #Se obtiene el HTML del producto
    page = requests.get(url, headers=headers, verify=False)
    soup1 = BeautifulSoup(page.content, "html.parser")

    #Se extrae el precio del HTML
    try:
        price = soup1.find('div', {"class": "product-price es__styling is--current-price css-11s12ax"}).text.strip()
               
    except:
        try:
            price = soup1.find("div", {"class" : "product-price is--current-price css-1ydfahe"}).text.strip() 
                
        except AttributeError:
            price = "No se ha podido obtener el precio para este producto."
        
    return price

def actual_date():
    now = datetime.now()
    date_now = now.date()
   
    return date_now

def actual_hour():
    now = datetime.now()
    hour_now = now.time()

    return hour_now

## Configuracion del Excel

In [3]:
#Incluir la localizacion del fichero excel con los datos de Nike:
directory = os.getcwd()
path = directory + "\\Precios.xlsx"

# Se abre el workbook y la hoja seleccionados (donde se guardo el documento por ultima vez)
wb_obj = openpyxl.load_workbook(path)
sheet_obj = wb_obj.active

# Obtener el numero de filas y columnas usadas
row_limit = sheet_obj.max_row
column_limit = sheet_obj.max_column
  
print("Total de filas:", row_limit)
print("Total de columnas:", column_limit)

#Introducir el numero de columna en la que se encuentra el SKU (A=1, B=2, C=3...)
sku_col = 1

#Introducir el numero de la fila en la que se encuentra el primer valor de sku
first_sku_row = 2

#Introducir el numero de columna en la que se encuentra el precio
precio_col = 4

#Introducir columna de pais
country_col = 5



Total de filas: 7400
Total de columnas: 5


In [4]:
# Cerrar el fichero Excel antes de ejecutar esta celda

#Introducir el formato de la web que se va a buscar
web_prefix = "https://www.nike.com/es/w?q="
web_prefix1 = "https://www.nike.com/pt/w?q="
web_prefix2 = "https://www.nike.com/fr/w?q="
pais1= "España" 
pais2= "Portugal" 
pais3= "Francia" 

date = actual_date()
hour = actual_hour()
time1 = time.time()

print(("INICIADO EL {} A LAS {} HORAS").format(date, hour))


for i in range(first_sku_row, row_limit + 1): 
    cell_obj = sheet_obj.cell(row = i, column = sku_col)
    sku = str(cell_obj.value)
    
    if sku == None:
        sku = ""
    
    #sku = str(sku).zfill(12)
    try:
        url = web_prefix+sku
        price = get_Nike_price(url)
        print(("{}: {} tiene un precio de: {} -> {}").format(str(i-1).zfill(4), sku, price, pais1))
        
        #Se introduce el precio en el Excel
        c1 = sheet_obj.cell(row = i, column = precio_col)
        c2 = sheet_obj.cell(row = i, column = country_col)
        c1.value = price
        c2.value = pais1
    except:
        try:
            url = web_prefix1+sku
            price = get_Nike_price(url)
            print(("{}: {} tiene un precio de: {} -> {}").format(str(i-1).zfill(4), sku, price, pais2))
    
            #Se introduce el precio en el Excel
            c1 = sheet_obj.cell(row = i, column = precio_col)
            c2 = sheet_obj.cell(row = i, column = country_col)
            c1.value = price
            c2.value = pais2
        except:
            try:
                url = web_prefix2+sku
                price = get_Nike_price2(url)
                print(("{}: {} tiene un precio de: {} -> {}").format(str(i-1).zfill(4), sku, price, pais3))
        
                #Se introduce el precio en el Excel
                c1 = sheet_obj.cell(row = i, column = precio_col)
                c2 = sheet_obj.cell(row = i, column = country_col)
                c1.value = price
                c2.value = pais3
            except:
                break
#Se guarda el fichero Excel con los cambios aplicados
wb_obj.save(path)
time2 = time.time()
hour1 = actual_hour()
dif_time = time2 - time1
print(("FINALIZADO EL {} A LAS {} HORAS").format(date, hour1))
print(("REALIZADO EN: {} horas").format((dif_time/60)/60))
print("ARCHIVO GUARDADO / PROGRAMA FINALIZADO")

INICIADO EL 2024-03-13 A LAS 11:32:50.785522 HORAS
0001: DO2792-010 tiene un precio de: 39,99 € -> España
0002: DA6925-012 tiene un precio de: 22,99 € -> España
0003: FB5110-340 tiene un precio de: 62,97 € -> España
0004: FN4547-133 tiene un precio de: No se ha podido obtener el precio para este producto. -> Francia
